In [ ]:
# ═══════════════════════════════ CELL 1: Mount + Shapefile ═══════════════════════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/high_plains_quifer.zip'
extract_path = '/content/ogallala_shp'
if not os.path.exists('/content/ogallala_shp/high_plains_quifer/hp_bound2010.shp'):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
print("Shapefile ready:", os.path.exists('/content/ogallala_shp/high_plains_quifer/hp_bound2010.shp'))

Shapefile ready: True


In [ ]:
# ═══════════════════════════════ CELL 2: Install (pinned!) ═══════════════════════════════
# Pinning fsspec/s3fs/gcsfs to 2025.3.0: the 2026.7.0 fsspec pulled by default
# is INCOMPATIBLE and corrupts earthaccess downloads (truncated/empty .nc4 files).
# If Colab offers "RESTART RUNTIME" after this, do it, then continue below.
!pip install -q earthaccess netCDF4 geopandas shapely h5py "fsspec==2025.3.0" "s3fs==2025.3.0" "gcsfs==2025.3.0" 2>&1 | tail -2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 6.9 MB/s eta 0:00:00


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════ CELL 3: Full pipeline (schema-adaptive) ═══════════════════════════════
# -*- coding: utf-8 -*-
"""OCO-2 / OCO-3 L2 Lite SIF — Ogallala (High Plains Aquifer) ROI extractor.
Adapts to BOTH Lite SIF schemas:
  modern : Date/Time, SIF_757/771nm (+Correction), Land_Fraction, Sounding_ID, Solar_Zenith...
  reduced: Delta_Time only, SIF_740nm + Daily_SIF_740/757/771nm, SZA/VZA/SAz/VAz, no Land_Fraction
"""

import os, re, gc, glob, time, shutil, logging, hashlib
from datetime import datetime, timedelta, time as dtime
import numpy as np
import h5py
import netCDF4 as nc4
import geopandas as gpd
import shapely
import earthaccess

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-7s | %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('ocosif')

# ─── CONFIGURATION ──────────────────────────────────────────────────────────
SHORT_NAME = "OCO2_L2_Lite_SIF"   # OCO-2.  For OCO-3: "OCO3_L2_Lite_SIF" (data from 2019-08)
VERSION    = None                 # e.g. "11.1r"; None = latest
MODE       = "points"             # "points" = per-sounding records | "grid" = daily lat/lon binning
GRID_RES   = 0.05                 # deg, MODE='grid' only

START_DATE = datetime(2014, 9, 6)

QUALITY_GOOD_ONLY  = True         # Lite SIF convention: Quality_Flag==0 is good
LAND_FRACTION_MIN  = 0.8          # applied only when Land_Fraction exists in the file
ADD_SIF_CORRECTION = True         # sif_757_corrected = SIF_757nm + SIF_Correction_757nm (when present)
PREFLIGHT          = True

SHAP_PATH      = '/content/ogallala_shp/high_plains_quifer/hp_bound2010.shp'
DRIVE_OUT_DIR  = '/content/drive/MyDrive/OCOSIF_L2_Ogallala'
LOCAL_WORK     = '/content/OCO_raw'
LOCAL_NC       = '/content/OCO_nc'

BATCH_DATES    = 150
FLUSH_DATES    = 25
BACKUP_DATES   = 75
MAX_RETRIES    = 4
N_THREADS      = 8
COMPRESS_LEVEL = 3
CHECKPOINT_HR  = 3

FILL_F4 = np.float32(-9999.0)
FILL_I1, FILL_I4, FILL_I8 = np.int8(-1), np.int32(-2147483647), np.int64(-9223372036854775807)
EPOCH      = datetime(1970, 1, 1)
TIME_UNITS = 'seconds since 1970-01-01 00:00:00'
TAI93_OFFSET = 725846400.0        # seconds 1970-01-01 → 1993-01-01 (UTC)

# Candidate variable names per logical field (schemas differ between versions)
FLOAT_VARS = {
    'sif_757':       ['SIF_757nm'],
    'sif_757_unc':   ['SIF_Uncertainty_757nm'],
    'sif_757_corr':  ['SIF_Correction_757nm'],
    'sif_771':       ['SIF_771nm'],
    'sif_771_unc':   ['SIF_Uncertainty_771nm'],
    'sif_771_corr':  ['SIF_Correction_771nm'],
    'sif_740':       ['SIF_740nm'],
    'sif_740_unc':   ['SIF_Uncertainty_740nm'],
    'sif_757_daily': ['Daily_SIF_757nm'],
    'sif_771_daily': ['Daily_SIF_771nm'],
    'sif_740_daily': ['Daily_SIF_740nm'],
    'land_fraction': ['Land_Fraction'],
    'solar_zenith':  ['Solar_Zenith', 'SZA'],
    'sensor_zenith': ['Sensor_Zenith', 'VZA'],
}
SIF_KEYS = ['sif_757', 'sif_771', 'sif_740', 'sif_757_daily', 'sif_771_daily', 'sif_740_daily']
INT_VARS = {'quality_flag': ['Quality_Flag'], 'orbit': ['Orbit', 'orbit_number']}
SID_CANDIDATES = ['Sounding_ID', 'sounding_id']

for d in (LOCAL_WORK, LOCAL_NC, DRIVE_OUT_DIR):
    os.makedirs(d, exist_ok=True)

# ─── UTILITIES ──────────────────────────────────────────────────────────────
def disk_free_gb(path='/'):
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize / 2**30

def load_roi(shp_path, deg=0.01):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    roi = gdf.union_all() if hasattr(gdf, 'union_all') else gdf.unary_union
    return roi.simplify(deg, preserve_topology=True)

def get_md5(filepath):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while chunk := f.read(10485760):
            h.update(chunk)
    return h.hexdigest()

def robust_drive_copy(src, dst):
    tmp = dst + '.tmp'
    src_md5 = get_md5(src)
    last_err = None
    for attempt in range(1, 4):
        try:
            with open(src, 'rb') as fs, open(tmp, 'wb') as fd:
                shutil.copyfileobj(fs, fd, length=10485760)
                fd.flush(); os.fsync(fd.fileno())
            if os.path.getsize(src) == os.path.getsize(tmp) and src_md5 == get_md5(tmp):
                os.replace(tmp, dst)
                log.info(f"Drive copy verified ({os.path.getsize(dst)/1048576:.1f} MiB)")
                return True
            raise IOError("Size or MD5 mismatch")
        except Exception as e:
            last_err = e
            log.error(f"Drive copy attempt {attempt} failed: {e}")
            if os.path.exists(tmp):
                try: os.remove(tmp)
                except OSError: pass
            time.sleep(5 * attempt)
    raise RuntimeError(f"Drive copy failed after 3 attempts: {last_err}")

# ─── LITE FILE READER (netCDF4 with h5py fallback) ──────────────────────────
class LiteFile:
    def __init__(self, path):
        self.backend, self._nc, self._h5 = None, None, None
        try:
            self._nc = nc4.Dataset(path)
            self.backend = 'netcdf4'
        except Exception as ne:
            try:
                self._h5 = h5py.File(path, 'r')
                self.backend = 'h5py'
            except Exception as he:
                raise IOError(f"netCDF4: {ne} | h5py: {he}")

    def __enter__(self): return self
    def __exit__(self, *exc): self.close()

    def close(self):
        for attr in ('_nc', '_h5'):
            h = getattr(self, attr)
            if h is not None:
                try: h.close()
                except Exception: pass
                setattr(self, attr, None)

    @property
    def varnames(self):
        if self._nc is not None:
            return set(self._nc.variables)
        return {k for k, v in self._h5.items() if isinstance(v, h5py.Dataset)}

    def attr(self, name, var):
        try:
            if self._nc is not None:
                return getattr(self._nc.variables[var], name, '')
            a = self._h5[var].attrs.get(name)
            if a is not None:
                a = np.atleast_1d(a)[0]
                return a.decode() if isinstance(a, bytes) else str(a)
        except Exception:
            pass
        return ''

    def read(self, name):
        """Float64 array with fill values mapped to NaN."""
        if self._nc is not None:
            return np.ma.filled(np.ma.asarray(self._nc.variables[name][:]).astype('f8'), np.nan)
        d = self._h5[name]
        arr = np.asarray(d[()]).astype('f8')
        fv = None
        for k in ('_FillValue', 'missing_value'):
            if k in d.attrs:
                fv = np.atleast_1d(d.attrs[k])[0]; break
        if fv is not None:
            arr[arr == fv] = np.nan
        sf, ao = d.attrs.get('scale_factor'), d.attrs.get('add_offset')
        if sf is not None: arr = arr * float(np.atleast_1d(sf)[0])
        if ao is not None: arr = arr + float(np.atleast_1d(ao)[0])
        return arr

def _read_float(f, names, n):
    for nm in names:
        if nm in f.varnames:
            try:
                a = f.read(nm)
                if a.size == n: return a
            except Exception: pass
    return None

def _read_int(f, names, n):
    return _read_float(f, names, n)      # ints arrive as f8, fills as NaN

def _read_mode(f, n):
    for nm in ('Sounding_Mode', 'sounding_mode', 'SoundingMode', 'Observation_Mode'):
        if nm not in f.varnames: continue
        try:
            raw = np.asarray(f.read(nm)) if False else None
            if f._nc is not None: raw = np.asarray(f._nc.variables[nm][:])
            else:                 raw = np.asarray(f._h5[nm][()])
            if raw.ndim == 2 and raw.dtype.kind == 'S':
                raw = nc4.chartostring(raw)
            raw = np.asarray(raw).ravel()
            if raw.dtype.kind == 'S':
                raw = np.char.decode(raw, 'utf-8', 'ignore')
            elif raw.dtype == object:
                raw = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in raw])
            raw = raw.astype('U')
            if raw.size != n: continue
            uniq, inv = np.unique(raw, return_inverse=True)
            return inv.astype('i1'), [str(u).strip().replace(' ', '_') for u in uniq]
        except Exception:
            continue
    return None, None

# ─── TIME RESOLUTION (handles missing Date/Time) ────────────────────────────
def parse_date_from_name(bn):
    """oco2_LtSIF_140906_B11012Ar_230329214756s.nc4 → 2014-09-06"""
    m = re.search(r'LtSIF_(\d{6})_', bn, re.I)
    if not m:
        m = re.search(r'_(\d{6})_', bn)
    if not m:
        return None
    s = m.group(1)
    try:
        return datetime(2000 + int(s[:2]), int(s[2:4]), int(s[4:6]))
    except ValueError:
        return None

def resolve_time(f, bn):
    """Return unix-seconds array, or None if no usable time info. Epoch-proof:
      1. 'Date' (+optional 'Time') present → absolute, used as-is (modern schema).
      2. Only 'Delta_Time' → RELATIVE intra-day offsets anchored to the filename
         date at 12:00 UTC (Lite SIF granules are single-day files, so this is as
         accurate as absolute time and immune to TAI93/GPS/unix epoch ambiguity).
    """
    if 'Date' in f.varnames:
        d = f.read('Date')
        if np.isfinite(d).any():
            sec = d * 86400.0
            if 'Time' in f.varnames:
                try:
                    t = f.read('Time'); med = float(np.nanmedian(t))
                    if np.isfinite(med) and 0.0 <= med < 86400.0:
                        sec = sec + t
                    elif np.isfinite(med) and med > 2 * 86400.0:
                        sec = t / 1000.0
                except Exception:
                    pass
            log.info("Time source: Date (+Time) — absolute")
            return sec

    day = parse_date_from_name(bn)
    if day is None:
        return None
    anchor = (day - EPOCH).total_seconds() + 43200.0        # 12:00 UTC

    if 'Delta_Time' in f.varnames:
        dt = f.read('Delta_Time')
        fin = dt[np.isfinite(dt)]
        if fin.size:
            spread = float(fin.max() - fin.min())
            scale = 1.0
            if 0.0 < spread <= 2.0:      # offsets in days
                scale = 86400.0
            elif spread >= 1.0e7:        # offsets in milliseconds
                scale = 1.0e-3
            med = float(np.median(fin))
            t_rel = np.where(np.isfinite(dt), (dt - med) * scale, 0.0)
            t_rel = np.clip(t_rel, -43200.0, 43200.0)           # stay inside the granule's day
            log.info(f"Time source: {day:%Y-%m-%d} 12:00 UTC + Delta_Time offsets "
                     f"(spread={spread:.4g} → scale {scale:g}; units attr: "
                     f"'{f.attr('units', 'Delta_Time') or 'none'}')")
            return anchor + t_rel
    return None    # extract_points falls back to filename noon for all soundings

# ─── VALIDATION ─────────────────────────────────────────────────────────────
def _file_kind(path):
    try:
        with open(path, 'rb') as fh:
            head = fh.read(512)
    except OSError:
        return 'unreadable'
    if head[:4] == b'\x89HDF': return 'hdf5'
    if head[:3] == b'CDF':     return 'netcdf'
    if not head:               return 'empty'
    if b'<' in head[:64] or b'html' in head.lower() or b'Error' in head:
        return 'html'
    return 'unknown'

def validate_nc(filepath):
    """Returns (ok, day, reason). Accepts BOTH schemas."""
    bn = os.path.basename(filepath)
    try:
        size = os.path.getsize(filepath)
    except OSError as e:
        return False, None, f'stat failed: {e}'
    if size == 0:   return False, None, '0 bytes'
    if size < 1024: return False, None, f'too small ({size} B)'
    kind = _file_kind(filepath)
    if kind == 'html':
        return False, None, 'HTML error page, not data (auth/redirect problem)'
    if kind not in ('netcdf', 'hdf5'):
        return False, None, f'unknown file type ({kind}) — transfer/auth problem'
    try:
        with LiteFile(filepath) as f:
            names = f.varnames
            missing = [v for v in ('Latitude', 'Longitude') if v not in names]
            if missing:
                return False, None, f'missing {missing}; has {sorted(names)}'
            if not any(v.upper().startswith('SIF_') for v in names):
                return False, None, f'no SIF_* variable; has {sorted(names)}'
            if ('Date' not in names) and ('Delta_Time' not in names):
                return False, None, f'no Date/Delta_Time variable; has {sorted(names)}'
            if 'Date' in names:
                d = f.read('Date')
                day = (EPOCH + timedelta(days=float(np.floor(np.nanmin(d))))).date()
            else:
                day = parse_date_from_name(bn)
                if day is None:
                    return False, None, 'no Date variable and filename date unparseable'
        return True, day, 'ok'
    except Exception as e:
        return False, None, f'{kind} open/read failed: {type(e).__name__}: {e}'

# ─── DIAGNOSTICS ────────────────────────────────────────────────────────────
def _auth_session():
    try:
        s = earthaccess.auth.get_session()
        if s is not None: return s
    except Exception: pass
    import requests
    return requests.Session()

def _https_link(result):
    try:
        links = result.data_links()
    except Exception:
        return None
    for u in links:
        if u.startswith('http') and 'opendap' not in u.lower() and '.nc' in u.lower():
            return u.split('?')[0]
    for u in links:
        if u.startswith('http'):
            return u.split('?')[0]
    return None

def _https_download(session, url, dst):
    tmp = dst + '.part'
    with session.get(url, stream=True, timeout=(30, 300), allow_redirects=True) as r:
        r.raise_for_status()
        ct = (r.headers.get('Content-Type') or '').lower()
        if 'html' in ct or ct.startswith('text/'):
            raise IOError(f'server returned {ct} — not data')
        cl = r.headers.get('Content-Length')
        with open(tmp, 'wb') as fo:
            for chunk in r.iter_content(8 << 20):
                if chunk: fo.write(chunk)
    if cl and os.path.getsize(tmp) != int(cl):
        os.remove(tmp); raise IOError(f'size mismatch: got {os.path.getsize(tmp)}, expected {cl}')
    os.replace(tmp, dst)

def inspect_file(path):
    size = os.path.getsize(path)
    log.info(f"Inspecting {os.path.basename(path)} ({size:,} B)")
    with open(path, 'rb') as fh:
        head = fh.read(300)
    if head[:4] == b'\x89HDF':
        log.info("Magic bytes: HDF5/netCDF-4 — header looks valid")
    elif head[:3] == b'CDF':
        log.info("Magic bytes: classic netCDF")
    elif head.lstrip()[:1] == b'<':
        log.error(f"Magic bytes: HTML — ERROR PAGE, not data! First bytes:\n{head[:200]!r}")
        return
    else:
        log.error(f"Magic bytes UNKNOWN: {head[:32]!r}")
        return
    try:
        with LiteFile(path) as f:
            log.info(f"Opened with backend '{f.backend}'. {len(f.varnames)} variables.")
            log.info(f"Variables: {sorted(f.varnames)}")
            if 'Date' in f.varnames:
                d = f.read('Date')
                log.info(f"Date range: {np.nanmin(d):.3f} … {np.nanmax(d):.3f} (days since 1970)")
            elif 'Delta_Time' in f.varnames:
                dt = f.read('Delta_Time')
                log.info(f"Delta_Time median={np.nanmedian(dt):.3f}, units='{f.attr('units', 'Delta_Time')}'")
    except Exception as e:
        log.error(f"Header OK but reading failed: {type(e).__name__}: {e}")
        log.error("→ likely TRUNCATED download (disk full or interrupted transfer).")

def run_diagnostics(sample_result=None, reason=None):
    log.info("═" * 72)
    log.info("DIAGNOSTICS — single-granule deep dive")
    log.info(f"Disk free: {disk_free_gb():.1f} GiB")
    log.info(f"Earthdata authenticated: {bool(getattr(earthaccess.auth, 'authenticated', False))}")
    if reason:
        log.info(f"Last validation failure: {reason}")
    res = sample_result
    if res is None:
        res = earthaccess.search_data(short_name=SHORT_NAME,
                                      temporal=(START_DATE.strftime('%Y-%m-%d'),
                                               (START_DATE + timedelta(days=10)).strftime('%Y-%m-%d')),
                                      count=1)
        res = res[0] if res else None
    if res is None:
        log.error("No granule found to diagnose — check SHORT_NAME / VERSION / dates.")
        return
    bn = result_basename(res) or 'sample.nc4'
    url = _https_link(res)
    log.info(f"Granule: {bn}")
    log.info(f"HTTPS link: {url}")
    try:
        r = _auth_session().head(url, timeout=60, allow_redirects=True)
        log.info(f"HEAD → HTTP {r.status_code} | Content-Type={r.headers.get('Content-Type')} "
                 f"| Content-Length={r.headers.get('Content-Length')}")
        if r.status_code in (401, 403):
            log.error("→ 401/403: session expired or GES DISC app not authorized.")
        if 'html' in (r.headers.get('Content-Type') or '').lower():
            log.error("→ HTML response: you are NOT reaching the data file.")
    except Exception as e:
        log.error(f"HEAD request failed: {e}")
    p = os.path.join(LOCAL_WORK, bn)
    if os.path.exists(p):
        try: os.remove(p)
        except OSError: pass
    try:
        log.info(f"earthaccess.download → {earthaccess.download([res], LOCAL_WORK, threads=1)}")
    except Exception as e:
        log.error(f"earthaccess.download failed: {e}")
    if os.path.exists(p):
        inspect_file(p)
    log.info("Now trying direct HTTPS (requests) download of the same file...")
    if os.path.exists(p):
        try: os.remove(p)
        except OSError: pass
    try:
        _https_download(_auth_session(), url, p)
        log.info(f"HTTPS download OK: {os.path.getsize(p):,} B")
        inspect_file(p)
    except Exception as e:
        log.error(f"HTTPS download failed: {e}")
    log.info("═" * 72)

# ─── EXTRACTION ─────────────────────────────────────────────────────────────
def extract_points(filepath, roi, bn=None):
    """Point-in-polygon ROI subset + QC of one daily granule. Returns rec dict or None."""
    bn = bn or os.path.basename(filepath)
    with LiteFile(filepath) as f:
        names = f.varnames
        for v in ('Latitude', 'Longitude'):
            if v not in names:
                raise IOError(f"{bn}: missing {v}")
        lat, lon = f.read('Latitude'), f.read('Longitude')
        n = lat.size

        sec = resolve_time(f, bn)
        if sec is None:
            day = parse_date_from_name(bn)
            if day is None:
                log.warning(f"{bn}: no usable time information; granule skipped")
                return None
            sec = np.full(n, (day - EPOCH).total_seconds() + 43200.0)
            log.warning(f"{bn}: no time variable — assigned 12:00 UTC on {day:%Y-%m-%d}")
        sec = np.where(np.isfinite(sec), sec, np.nan)

        keep = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(sec)
        minx, miny, maxx, maxy = roi.bounds
        keep &= (lon >= minx) & (lon <= maxx) & (lat >= miny) & (lat <= maxy)

        q  = _read_int(f, INT_VARS['quality_flag'], n)
        lf = _read_float(f, FLOAT_VARS['land_fraction'], n)
        if lf is not None and np.nanmax(lf) > 1.5: lf = lf / 100.0
        if QUALITY_GOOD_ONLY and q is not None:
            keep &= (q == 0)
        if LAND_FRACTION_MIN is not None and lf is not None:
            keep &= (np.nan_to_num(lf, nan=0.0) >= LAND_FRACTION_MIN)

        idx = np.where(keep)[0]
        if idx.size == 0: return None

        rec = {'n': int(idx.size), 'time': sec[idx],
               'lat': lat[idx].astype('f4'), 'lon': lon[idx].astype('f4')}
        for key, cands in FLOAT_VARS.items():
            a = _read_float(f, cands, n)
            if a is not None: rec[key] = a[idx].astype('f4')
        if 'land_fraction' not in rec and lf is not None:
            rec['land_fraction'] = lf[idx].astype('f4')
        sa = _read_float(f, ['Solar_Azimuth', 'SAz'], n)
        va = _read_float(f, ['Sensor_Azimuth', 'VAz', 'Observatory_Azimuth'], n)
        if sa is not None and va is not None:
            rel = np.abs(sa - va) % 360.0
            rec['rel_azimuth'] = np.where(rel > 180.0, 360.0 - rel, rel)[idx].astype('f4')
        for key, cands in INT_VARS.items():
            a = _read_int(f, cands, n)
            if a is not None:
                rec[key] = np.where(np.isfinite(a[idx]), a[idx], -1.0).astype('i4' if key == 'orbit' else 'i1')
        sid = _read_float(f, SID_CANDIDATES, n)
        if sid is not None:
            rec['sounding_id'] = np.where(np.isfinite(sid[idx]), sid[idx], -1.0).astype('i8')
        codes, mapping = _read_mode(f, n)
        if codes is not None:
            rec['mode_code'], rec['mode_mapping'] = codes[idx], mapping
        if ADD_SIF_CORRECTION and 'sif_757' in rec and 'sif_757_corr' in rec:
            rec['sif_757_corrected'] = (rec['sif_757'] + rec['sif_757_corr']).astype('f4')
    return rec

# ─── NETCDF WRITERS ─────────────────────────────────────────────────────────
def create_points_nc(path, rec):
    ds = nc4.Dataset(path, 'w', format='NETCDF4')
    ds.createDimension('obs', None)
    ds.Conventions = 'CF-1.8'; ds.featureType = 'point'
    ds.title = f'{SHORT_NAME} — Ogallala ROI soundings'
    ds.source = f'NASA {SHORT_NAME} L2 Lite SIF'
    ds.history = f'Created {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}'
    ds.comment = (f'QC: Quality_Flag==0={QUALITY_GOOD_ONLY}, '
                  f'Land_Fraction>={LAND_FRACTION_MIN} (when variable present). '
                  f'SIF units W m-2 sr-1 um-1. sif_757_corrected = SIF_757nm + SIF_Correction_757nm.')

    def _mk(name, dtype, fill, **attrs):
        v = ds.createVariable(name, dtype, ('obs',), fill_value=fill,
                              zlib=True, complevel=COMPRESS_LEVEL, chunksizes=(65536,))
        for k, val in attrs.items(): v.setncattr(k, val)
        return v

    _mk('time', 'f8', None, units=TIME_UNITS, calendar='proleptic_gregorian',
        standard_name='time', long_name='sounding time (UTC)')
    _mk('lat', 'f4', FILL_F4, units='degrees_north', standard_name='latitude')
    _mk('lon', 'f4', FILL_F4, units='degrees_east',  standard_name='longitude')

    dmap = {'int8': ('i1', FILL_I1), 'int16': ('i2', np.int16(-32767)),
            'int32': ('i4', FILL_I4), 'int64': ('i8', FILL_I8)}
    for k, arr in rec.items():
        if k in ('n', 'time', 'lat', 'lon', 'mode_mapping'): continue
        if arr.dtype.kind in 'iu':
            dt, fill = dmap.get(arr.dtype.name, ('i4', FILL_I4))
        else:
            dt, fill = 'f4', FILL_F4
        attrs = {'coordinates': 'time lat lon'}
        if k.startswith('sif'):  attrs['units'] = 'W m-2 sr-1 um-1'
        if k == 'land_fraction': attrs['units'] = '1'
        if k == 'mode_code':
            mapping = rec.get('mode_mapping') or ['UNKNOWN']
            attrs['flag_values'] = np.arange(len(mapping), dtype='i1')
            attrs['flag_meanings'] = ' '.join(mapping)
            attrs['long_name'] = 'OCO sounding observation mode'
        _mk(k, dt, fill, **attrs)
    return ds

def append_points(ds, rec):
    n = rec['n']
    if n == 0: return 0
    n0 = ds.variables['time'].size
    for k, arr in rec.items():
        if k in ('n', 'mode_mapping'): continue
        var = ds.variables.get(k)
        if var is None or not isinstance(arr, np.ndarray): continue
        if var.dtype.kind == 'f':
            arr = np.where(np.isfinite(arr), arr, FILL_F4).astype(var.dtype)
        else:
            arr = arr.astype(var.dtype)
        var[n0:n0 + n] = arr
    return n

def build_grid(roi, res):
    minx, miny, maxx, maxy = roi.bounds
    lat0, lat1 = np.floor(miny/res)*res, np.ceil(maxy/res)*res
    lon0, lon1 = np.floor(minx/res)*res, np.ceil(maxx/res)*res
    return np.arange(lat0 + res/2, lat1, res), np.arange(lon0 + res/2, lon1, res)

def create_grid_nc(path, lat_c, lon_c, sif_keys):
    ny, nx = lat_c.size, lon_c.size
    ds = nc4.Dataset(path, 'w', format='NETCDF4')
    for d, s in (('time', None), ('y', ny), ('x', nx)): ds.createDimension(d, s)
    t = ds.createVariable('time', 'f8', ('time',))
    t.units = TIME_UNITS; t.calendar = 'proleptic_gregorian'; t.standard_name = 'time'
    lat = ds.createVariable('lat', 'f4', ('y',), zlib=True, complevel=1)
    lat[:], lat.units, lat.standard_name = lat_c.astype('f4'), 'degrees_north', 'latitude'
    lon = ds.createVariable('lon', 'f4', ('x',), zlib=True, complevel=1)
    lon[:], lon.units, lon.standard_name = lon_c.astype('f4'), 'degrees_east', 'longitude'
    chunk = (1, min(ny, 512), min(nx, 512))
    for k in sif_keys:
        for suf in ('mean', 'std'):
            var = ds.createVariable(f'{k}_{suf}', 'f4', ('time', 'y', 'x'), fill_value=FILL_F4,
                                    zlib=True, complevel=COMPRESS_LEVEL, chunksizes=chunk)
            var.units = 'W m-2 sr-1 um-1'; var.coordinates = 'lat lon'
    cnt = ds.createVariable('n_obs', 'i4', ('time', 'y', 'x'), zlib=True,
                            complevel=COMPRESS_LEVEL, chunksizes=chunk)
    cnt.long_name = 'soundings averaged per cell per day'; cnt.coordinates = 'lat lon'
    crs = ds.createVariable('crs', 'i4')
    crs.grid_mapping_name = 'latitude_longitude'
    crs.semi_major_axis = 6378137.0; crs.inverse_flattening = 298.257223563
    ds.Conventions = 'CF-1.8'
    ds.title = f'{SHORT_NAME} — Ogallala ROI daily gridded SIF ({GRID_RES} deg)'
    ds.comment = 'Daily mean of Quality_Flag==0 soundings per cell.'
    return ds

def append_grid_day(ds, day_sec, lat, lon, fields, lat_c, lon_c):
    res, ny, nx = GRID_RES, lat_c.size, lon_c.size
    iy = np.floor((lat - (lat_c[0] - res/2)) / res).astype('i8')
    ix = np.floor((lon - (lon_c[0] - res/2)) / res).astype('i8')
    inside = (iy >= 0) & (iy < ny) & (ix >= 0) & (ix < nx)
    flat = iy[inside] * nx + ix[inside]

    t = ds.variables['time'].size
    if t > 0 and abs(float(ds.variables['time'][t-1]) - day_sec) < 1.0:
        t -= 1
    ds.variables['time'][t] = day_sec

    keys = [v[:-5] for v in ds.variables if v.endswith('_mean')]
    for k in keys:
        mean = np.full(ny*nx, np.nan); std = np.full(ny*nx, np.nan)
        vals = fields.get(k)
        if vals is not None:
            v = np.asarray(vals, 'f8')[inside]; m = np.isfinite(v)
            if m.any():
                fl = flat[m]; c = np.bincount(fl, minlength=ny*nx).astype('f8')
                mean = np.where(c > 0, np.bincount(fl, weights=v[m], minlength=ny*nx)/np.maximum(c,1), np.nan)
                ssq  = np.bincount(fl, weights=v[m]**2, minlength=ny*nx)
                std  = np.sqrt(np.where(c > 0, np.clip(ssq/np.maximum(c,1) - mean**2, 0, None), np.nan))
        ds.variables[f'{k}_mean'][t] = np.where(np.isfinite(mean), mean, FILL_F4).reshape(ny, nx).astype('f4')
        ds.variables[f'{k}_std'][t]  = np.where(np.isfinite(std),  std,  FILL_F4).reshape(ny, nx).astype('f4')
    cnt = np.bincount(flat, minlength=ny*nx).reshape(ny, nx) if flat.size else np.zeros((ny, nx), 'i8')
    ds.variables['n_obs'][t] = cnt.astype('i4')

def granule_grid_records(filepath, roi):
    rec = extract_points(filepath, roi)
    if rec is None: return []
    day_idx = np.floor(rec['time'] / 86400.0).astype('i8')
    out = []
    for d in np.unique(day_idx):
        m = day_idx == d
        fields = {k: rec[k][m] for k in SIF_KEYS if k in rec}
        out.append((float(d * 86400), rec['lat'][m].astype('f8'),
                    rec['lon'][m].astype('f8'), fields))
    return out

# ─── MANIFEST (resume) ──────────────────────────────────────────────────────
def manifest_add(path, basename, day):
    with open(path, 'a') as fh:
        fh.write(f'{basename}\t{day:%Y-%m-%d}\n'); fh.flush(); os.fsync(fh.fileno())

def load_manifest(path):
    out = {}
    if os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    try: out[parts[0]] = datetime.strptime(parts[1], '%Y-%m-%d').date()
                    except ValueError: pass
    return out

# ─── DOWNLOAD + VALIDATE + RETRY (+ HTTPS FALLBACK) ─────────────────────────
def result_basename(r):
    try:
        for u in r.data_links():
            if '.nc' in os.path.basename(u):
                return os.path.basename(u.split('?')[0])
        return os.path.basename(r.data_links()[0].split('?')[0])
    except Exception:
        return None

def repair_via_https(pairs):
    s = _auth_session()
    fixed = 0
    for bn, r in pairs:
        url = _https_link(r)
        p = os.path.join(LOCAL_WORK, bn)
        if url is None:
            log.warning(f"No https link for {bn}; cannot HTTPS-fallback"); continue
        for a in range(1, 3):
            try:
                _https_download(s, url, p)
                ok, day, why = validate_nc(p)
                if ok: fixed += 1; break
                log.warning(f"HTTPS attempt {a} for {bn}: {why}")
                if os.path.exists(p):
                    try: os.remove(p)
                    except OSError: pass
            except Exception as e:
                log.error(f"HTTPS attempt {a} for {bn} failed: {e}")
                if os.path.exists(p):
                    try: os.remove(p)
                    except OSError: pass
    log.info(f"HTTPS fallback recovered {fixed}/{len(pairs)} file(s)")
    return fixed

def download_and_validate(results):
    expected = {bn: r for bn, r in ((result_basename(r), r) for r in results) if bn}
    try:
        earthaccess.download(list(expected.values()), LOCAL_WORK, threads=N_THREADS)
    except Exception as e:
        log.error(f"Initial download error: {e}")

    batch, failures, logged = {}, {}, set()
    for attempt in range(1, MAX_RETRIES + 1):
        need = []
        for bn, r in expected.items():
            p = os.path.join(LOCAL_WORK, bn)
            if not os.path.exists(p):
                failures[bn] = 'missing after download'
                need.append((bn, r)); continue
            ok, day, why = validate_nc(p)
            if ok:
                batch[bn] = (day, p, bn); failures.pop(bn, None)
            else:
                failures[bn] = why
                need.append((bn, r))
                if why not in logged:
                    logged.add(why)
                    log.warning(f"Corrupt/unreadable: {bn} — {why}")

        if not need:
            log.info("All files validated successfully."); break

        if attempt >= MAX_RETRIES:
            log.error(f"Max retries reached. {len(need)} file(s) skipped. Example failures:")
            for bn in list(failures)[:5]:
                log.error(f"   {bn}: {failures[bn]}")
            break

        if attempt == 1 and len(need) == len(expected):
            log.error(f"ALL {len(need)} files failed validation — systemic problem. Running diagnostics...")
            run_diagnostics(sample_result=need[0][1], reason=failures.get(need[0][0]))
            log.warning("Retrying every file via direct HTTPS (bypasses earthaccess/fsspec)...")
            repair_via_https(need)
        elif attempt == 2:
            log.warning("Fallback: direct HTTPS for remaining file(s)...")
            repair_via_https(need)
        else:
            log.warning(f"Redownloading {len(need)} file(s) via earthaccess (attempt {attempt+1}/{MAX_RETRIES})...")
            for bn, _ in need:
                p = os.path.join(LOCAL_WORK, bn)
                if os.path.exists(p):
                    try: os.remove(p)
                    except OSError: pass
            try:
                earthaccess.download([r for _, r in need], LOCAL_WORK, threads=4)
            except Exception as e:
                log.error(f"earthaccess redownload failed: {e}"); time.sleep(10)
        time.sleep(2)

    out = sorted(batch.values(), key=lambda x: x[0])
    log.info(f"Proceeding with {len(out)}/{len(expected)} valid granule(s).")
    return out

# ─── PREFLIGHT ──────────────────────────────────────────────────────────────
def preflight(roi):
    log.info("PREFLIGHT: testing single-granule download + extraction...")
    t0 = START_DATE
    res = earthaccess.search_data(short_name=SHORT_NAME,
                                  temporal=(t0.strftime('%Y-%m-%d'),
                                            (t0 + timedelta(days=10)).strftime('%Y-%m-%d')),
                                  count=1)
    if not res:
        log.warning("Preflight search returned no granules — check SHORT_NAME/VERSION/temporal.")
        return True
    res = res[0]
    bn = result_basename(res) or 'sample.nc4'
    log.info(f"Preflight granule: {bn}")
    log.info(f"Link: {_https_link(res)}")
    p = os.path.join(LOCAL_WORK, bn)
    if os.path.exists(p):
        try: os.remove(p)
        except OSError: pass
    try:
        earthaccess.download([res], LOCAL_WORK, threads=1)
    except Exception as e:
        log.error(f"Preflight download failed: {e}")
    ok, day, why = (False, None, 'file missing')
    if os.path.exists(p):
        ok, day, why = validate_nc(p)
    if ok:
        log.info(f"Preflight OK: {bn} ({day})")
        try:
            rec = extract_points(p, roi, bn)
            log.info(f"Preflight extraction test: {rec['n'] if rec else 0} soundings in ROI — schema handled.")
        except Exception as e:
            log.warning(f"Preflight file valid, but extraction test failed: {e}")
        try: os.remove(p)
        except OSError: pass
        return True
    log.warning(f"Preflight failed: {why}. Trying direct HTTPS fallback...")
    try:
        _https_download(_auth_session(), _https_link(res), p)
        ok, day, why = validate_nc(p)
    except Exception as e:
        log.error(f"HTTPS fallback failed: {e}")
    if ok:
        log.info(f"Preflight OK via HTTPS fallback: {bn} ({day}).")
        try:
            rec = extract_points(p, roi, bn)
            log.info(f"Preflight extraction test: {rec['n'] if rec else 0} soundings in ROI.")
        except Exception as e:
            log.warning(f"Extraction test failed: {e}")
        try: os.remove(p)
        except OSError: pass
        return True
    log.error("═" * 72)
    log.error("PREFLIGHT FAILED — checklist:")
    log.error(" 1. Earthdata profile → Applications → 'NASA GESDISC DATA ARCHIVE' authorized")
    log.error(" 2. Re-login: earthaccess.login(strategy='interactive', persist=True)")
    log.error(" 3. Restart runtime, reinstall with pinned fsspec/s3fs (Cell 2)")
    log.error(" 4. Check disk: !df -h /content")
    log.error("═" * 72)
    run_diagnostics(sample_result=res, reason=why)
    return False

# ─── PIPELINE ───────────────────────────────────────────────────────────────
def run_pipeline():
    earthaccess.login(strategy="interactive", persist=True)
    roi = load_roi(SHAP_PATH)
    try: shapely.prepare(roi)
    except Exception: pass
    log.info(f"ROI bounds: {np.round(roi.bounds, 3)}")
    log.info(f"Disk free: {disk_free_gb():.1f} GiB")

    if PREFLIGHT and not preflight(roi):
        raise RuntimeError("Preflight failed — see diagnostics above. Fix and re-run run_pipeline().")

    out_name = f'{SHORT_NAME}_Ogallala_FULL.nc'
    local_nc = os.path.join(LOCAL_NC, out_name)
    manifest_path = os.path.join(LOCAL_NC, 'processed_manifest.txt')
    if not os.path.exists(manifest_path): open(manifest_path, 'w').close()

    if not os.path.exists(local_nc):
        for src, dst in ((os.path.join(DRIVE_OUT_DIR, out_name), local_nc),
                         (os.path.join(DRIVE_OUT_DIR, 'processed_manifest.txt'), manifest_path)):
            if os.path.exists(src):
                shutil.copy2(src, dst); log.info(f"Restored {os.path.basename(src)} from Drive")

    processed = load_manifest(manifest_path)
    grid = build_grid(roi, GRID_RES) if MODE == 'grid' else None

    start_dt = START_DATE
    if processed:
        start_dt = datetime.combine(max(processed.values()) + timedelta(days=1), dtime.min)
        log.info(f"Manifest resume: {len(processed)} granules done; starting {start_dt:%Y-%m-%d}")
    elif os.path.exists(local_nc):
        try:
            with nc4.Dataset(local_nc) as ds:
                last = float(ds.variables['time'][-1])
            start_dt = (EPOCH + timedelta(seconds=last)).replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
            log.warning(f"Manifest missing — resuming after {start_dt:%Y-%m-%d}. A partial last day may exist.")
        except Exception as e:
            log.warning(f"Existing NC unreadable ({e}); starting fresh.")
            os.remove(local_nc)

    next_checkpoint = time.time() + CHECKPOINT_HR * 3600
    since_flush = since_backup = 0

    while start_dt < datetime.now():
        end_dt = min(start_dt + timedelta(days=BATCH_DATES), datetime.now())
        log.info(f"\nBatch: {start_dt:%Y-%m-%d} → {end_dt:%Y-%m-%d} (disk free {disk_free_gb():.1f} GiB)")

        kw = dict(short_name=SHORT_NAME,
                  temporal=(start_dt.strftime('%Y-%m-%dT%H:%M:%SZ'), end_dt.strftime('%Y-%m-%dT%H:%M:%SZ')))
        if VERSION: kw['version'] = VERSION
        results = earthaccess.search_data(**kw)
        if not results:
            start_dt = end_dt + timedelta(days=1); continue

        batch = download_and_validate(results)
        if not batch:
            start_dt = end_dt + timedelta(days=1); continue

        ds = nc4.Dataset(local_nc, 'a') if os.path.exists(local_nc) else None
        processed_count = 0

        for day, path, bn in batch:
            if bn in processed:
                try: os.remove(path)
                except OSError: pass
                continue
            try:
                if MODE == 'points':
                    rec = extract_points(path, roi, bn)
                    if rec is None:
                        log.info(f"{bn}: 0 soundings in ROI")
                        manifest_add(manifest_path, bn, day); continue
                    if ds is None: ds = create_points_nc(local_nc, rec)
                    append_points(ds, rec)
                else:
                    recs = granule_grid_records(path, roi)
                    if not recs:
                        log.info(f"{bn}: 0 soundings in ROI")
                        manifest_add(manifest_path, bn, day); continue
                    if ds is None:
                        keys = [k for k in SIF_KEYS if k in recs[0][3]]
                        ds = create_grid_nc(local_nc, *grid, keys)
                    for day_sec, la, lo, fields in recs:
                        append_grid_day(ds, day_sec, la, lo, fields, *grid)
                manifest_add(manifest_path, bn, day)
                processed[bn] = day
                processed_count += 1; since_flush += 1
            except Exception as e:
                log.error(f"UNEXPECTED ERROR processing {bn}: {e}. Skipping.")
            finally:
                if os.path.exists(path):
                    try: os.remove(path)
                    except OSError: pass

            if ds is not None and time.time() >= next_checkpoint:
                log.info(f"{CHECKPOINT_HR}-hour runtime checkpoint reached.")
                ds.sync(); ds.close(); ds = None
                for f in (local_nc, manifest_path):
                    try: robust_drive_copy(f, os.path.join(DRIVE_OUT_DIR, os.path.basename(f)))
                    except Exception as e: log.error(f"Checkpoint copy failed: {e}")
                ds = nc4.Dataset(local_nc, 'a')
                next_checkpoint = time.time() + CHECKPOINT_HR * 3600
                gc.collect()

            if ds is not None and since_flush >= FLUSH_DATES:
                ds.sync(); since_flush = 0; since_backup += 1
                log.info(f"Flushed to local disk ({processed_count} granules this batch).")
                if since_backup >= BACKUP_DATES:
                    log.info("Backing up monolithic file to Drive...")
                    ds.close(); ds = None
                    for f in (local_nc, manifest_path):
                        robust_drive_copy(f, os.path.join(DRIVE_OUT_DIR, os.path.basename(f)))
                    ds = nc4.Dataset(local_nc, 'a')
                    since_backup = 0
                gc.collect()

        if ds is not None:
            ds.sync(); ds.close()
        log.info(f"Batch complete. Processed {processed_count} granules. Backing up to Drive...")
        for f in (local_nc, manifest_path):
            if os.path.exists(f):
                robust_drive_copy(f, os.path.join(DRIVE_OUT_DIR, os.path.basename(f)))

        for pat in ('*.nc4', '*.nc', '*.h5', '*.part'):
            for f in glob.glob(os.path.join(LOCAL_WORK, pat)):
                try: os.remove(f)
                except OSError: pass
        start_dt = end_dt + timedelta(days=1)
        gc.collect()

    log.info("Pipeline complete — all available granules processed.")

run_pipeline()

Enter your Earthdata Login username: watcher69
Enter your Earthdata password: ··········


/usr/local/lib/python3.13/dist-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/usr/local/lib/python3.13/dist-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


  0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/136 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/136 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/138 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/138 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/114 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/114 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/114 [00:00<?, ?it/s]